In [3]:
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained("ProsusAI/finbert")
model = BertForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    use_safetensors=True
)
model.eval()




BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [4]:
def finbert_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)[0]

    return {
        "positive": probs[0].item(),
        "negative": probs[1].item(),
        "neutral": probs[2].item()
    }



In [5]:
news_data = [
    {
        "date": "2024-01-02",
        "text": "Apple reported strong quarterly earnings beating expectations."
    },
    {
        "date": "2024-01-03",
        "text": "Apple stock fell as the company warned about declining iPhone sales."
    },
    {
        "date": "2024-01-04",
        "text": "Markets remained neutral as Apple announced no major updates."
    }
]


In [6]:
records = []

for item in news_data:
    scores = finbert_sentiment(item["text"])
    records.append({
        "date": item["date"],
        "text": item["text"],
        **scores
    })

sentiment_df = pd.DataFrame(records)
sentiment_df


,date,text,positive,negative,neutral
0,2024-01-02,Apple reported strong quarterly earnings beati...,0.953725,0.024255,0.022020
1,2024-01-03,Apple stock fell as the company warned about d...,0.008316,0.960573,0.031111
2,2024-01-04,Markets remained neutral as Apple announced no...,0.069917,0.417105,0.512978


In [7]:
sentiment_df.to_csv("../WEEK3/raw_sentiment.csv", index=False)
